# Part III: ABC XYZ Analysis

## Basic settings

In [57]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [58]:
import os
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
pd.options.display.max_rows = 50
pd.options.display.max_columns = None

In [59]:
DATA_DIR = "../data/processed"

## Load processed data

In [60]:
sku_metric = pd.read_csv(os.path.join(DATA_DIR, "sku_metric.csv"))
df_inventory = pd.read_csv(os.path.join(DATA_DIR, "inventory_processed.csv"), parse_dates = ["date"])

## ABC XYZ analysis

### ABC analysis

In [61]:
df_inventory['revenue'] = df_inventory['sales_quantity'] * df_inventory['unit_price']

In [62]:
abc_df = df_inventory.groupby("sku_id").agg(
    total_revenue = ("revenue", "sum")
).reset_index()
abc_df = abc_df.sort_values("total_revenue", ascending=False).reset_index(drop=True)

abc_df['cumulative_revenue'] = abc_df['total_revenue'].cumsum()
abc_df['cumulative_revenue_pct'] = abc_df['cumulative_revenue'] / abc_df['total_revenue'].sum()

In [63]:
def abc(x):
    if x <= 0.8:
        return 'A'
    elif x <= 0.95:
        return 'B'
    else:
        return 'C'

In [64]:
abc_df['abc_class'] = abc_df['cumulative_revenue_pct'].apply(abc)
abc_df['abc_class'].value_counts()

abc_class
A    73
C    39
B    38
Name: count, dtype: int64

### XYZ analysis

In [65]:
xyz_df = df_inventory.groupby("sku_id").agg(
    demand_std = ("demand", "std"),
    demand_mean = ("demand", "mean")
).reset_index()

xyz_df['cv'] = (xyz_df['demand_std'] / xyz_df['demand_mean']).replace([np.inf, -np.inf], np.nan)

In [66]:
xyz_df['cv'].describe()

count    150.000000
mean       0.454726
std        0.048992
min        0.384897
25%        0.416865
50%        0.439150
75%        0.486373
max        0.575805
Name: cv, dtype: float64

- CV ranges from 0.38 to 0.58 → limited differentiation across SKUs  
- Strong seasonality inflates CoV (captures pattern, not randomness)  
- CoV is ineffective for XYZ classification  
- Forecast error is used instead to measure demand uncertainty  

In [67]:
from statsforecast.models import Naive, SeasonalNaive, AutoTheta
from statsforecast import StatsForecast
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

split = '2025-10-01'
train = df_inventory[df_inventory['date'] < split]
test = df_inventory[df_inventory['date'] >= split]

In [68]:
df_train_sf = train[['sku_id', 'date', 'demand']].rename(columns = {
    'sku_id': 'unique_id',
    'date': 'ds',
    'demand': 'y'
})

test_df = test[['sku_id','date','demand']].rename(columns={'sku_id':'unique_id','date':'ds','demand':'y'})

h = test_df['ds'].nunique()

models = [
    Naive(),
    SeasonalNaive(season_length = 365),
    AutoTheta(season_length = 365)
]

sf = StatsForecast(models = models, freq = 'D', n_jobs = -1)
sf.fit(df_train_sf)
df_forecast = sf.predict(h = h)
df_forecast.head()

,unique_id,ds,Naive,SeasonalNaive,AutoTheta
0,SKU0001,2025-10-01,7.0,4.0,4.517584
1,SKU0001,2025-10-02,7.0,4.0,4.517408
2,SKU0001,2025-10-03,7.0,3.0,4.517228
3,SKU0001,2025-10-04,7.0,4.0,4.517045
4,SKU0001,2025-10-05,7.0,3.0,4.516860


In [69]:
def evaluate_models(y_true, predictions: dict):
    results = []

    eval_df = pd.DataFrame({'y_true': y_true})
    for model_name, y_pred in predictions.items():
        eval_df[model_name] = np.array(y_pred)

    mean_actual = eval_df.loc[eval_df['y_true'].notnull(), 'y_true'].mean()

    for model_name in predictions.keys():
        mask = eval_df['y_true'].notnull() & eval_df[model_name].notnull()
        y_true_arr = eval_df.loc[mask, 'y_true'].values
        y_pred_arr = eval_df.loc[mask, model_name].values

        mae   = mean_absolute_error(y_true_arr, y_pred_arr)
        rmse  = root_mean_squared_error(y_true_arr, y_pred_arr)
        bias  = np.mean(y_pred_arr - y_true_arr)

        mae_pct  = mae  / mean_actual * 100
        rmse_pct = rmse / mean_actual * 100
        bias_pct = bias / mean_actual * 100

        results.append({
            'Model' : model_name,
            'MAE'   : round(mae, 4),
            '%MAE'  : round(mae_pct, 2),
            'RMSE'  : round(rmse, 4),
            '%RMSE' : round(rmse_pct, 2),
            'Bias'  : round(bias, 4),
            '%Bias' : round(bias_pct, 2)
        })

    return pd.DataFrame(results).set_index('Model')

In [70]:
result = df_forecast.merge(test_df, on=['unique_id', 'ds'], how='left')
result.head()

,unique_id,ds,Naive,SeasonalNaive,AutoTheta,y
0,SKU0001,2025-10-01,7.0,4.0,4.517584,2
1,SKU0001,2025-10-02,7.0,4.0,4.517408,1
2,SKU0001,2025-10-03,7.0,3.0,4.517228,3
3,SKU0001,2025-10-04,7.0,4.0,4.517045,3
4,SKU0001,2025-10-05,7.0,3.0,4.516860,8


In [71]:
metric_eval = evaluate_models(
    y_true      = result['y'],
    predictions = {
        'Naive': result['Naive'],
        'SeasonalNaive': result['SeasonalNaive'],
        'AutoTheta': result['AutoTheta']
    }
)

print(metric_eval)

                  MAE   %MAE    RMSE  %RMSE    Bias  %Bias
Model                                                     
Naive          5.6783  45.90  7.6227  61.62 -2.2580 -18.25
SeasonalNaive  5.4260  43.86  7.2734  58.79 -1.2521 -10.12
AutoTheta      4.7096  38.07  6.4607  52.22 -2.6317 -21.27


In [72]:
def mae_pct(group):
    mae = mean_absolute_error(group['y'], group['AutoTheta'])
    return mae / np.mean(group['y'])

xyz_new = result.groupby('unique_id').apply(mae_pct).reset_index()
xyz_new.columns = ['sku_id', '%mae_autotheta']

In [73]:
xyz_new['%mae_autotheta'].describe()

count    150.000000
mean       0.395783
std        0.053851
min        0.283468
25%        0.353961
50%        0.394423
75%        0.426066
max        0.531968
Name: %mae_autotheta, dtype: float64

In [74]:
q1 = xyz_new['%mae_autotheta'].quantile(0.25)
q3 = xyz_new['%mae_autotheta'].quantile(0.75)

def classify_xyz(val):
    if val < q1:
        return 'X'
    elif val <= q3:
        return 'Y'
    else:
        return 'Z'

xyz_new['xyz_class'] = xyz_new['%mae_autotheta'].apply(classify_xyz)
xyz_new['xyz_class'].value_counts()

xyz_class
Y    74
Z    38
X    38
Name: count, dtype: int64

In [75]:
abc_xyz_df = abc_df[['sku_id', 'abc_class']].merge(xyz_new[['sku_id', 'xyz_class']], on = 'sku_id', how = 'left')
abc_xyz_df.head()

,sku_id,abc_class,xyz_class
0,SKU0057,A,X
1,SKU0055,A,X
2,SKU0121,A,X
3,SKU0007,A,X
4,SKU0028,A,X


In [76]:
abc_xyz_df['class'] = abc_xyz_df['abc_class'] + abc_xyz_df['xyz_class']

## Define problem

In [77]:
sku_class = sku_metric.merge(abc_xyz_df[['sku_id', 'class']], on = 'sku_id', how = 'left')
sku_class.head()

,sku_id,avg_inventory,total_demand,total_sales,num_days,daily_demand,DOI,fill_rate,lost_sales,CV,class
0,SKU0001,412.398085,4007,4007,731,5.481532,75.234090,1.000000,0,0.546062,CZ
1,SKU0002,283.041040,6392,6341,731,8.744186,32.369055,0.992021,51,0.497865,BY
2,SKU0003,173.990424,4866,4789,731,6.656635,26.137896,0.984176,77,0.528030,CZ
3,SKU0004,348.023256,12222,12222,731,16.719562,20.815333,1.000000,0,0.432078,BZ
4,SKU0005,464.596443,14919,14919,731,20.409029,22.764260,1.000000,0,0.419606,AX


In [78]:
issue_define = sku_class.groupby('class').agg(
    sku_count = ('sku_id', 'count'),
    avg_fill_rate = ('fill_rate', 'mean'),
    avg_lost_sales = ('lost_sales', 'mean'),
    avg_doi = ('DOI', 'mean')
).round(3).reset_index()


In [79]:
issue_define.head(9)

,class,sku_count,avg_fill_rate,avg_lost_sales,avg_doi
0,AX,26,0.958,780.500,14.452
1,AY,39,0.954,753.974,22.883
2,AZ,8,0.979,287.375,22.987
3,BX,4,0.984,241.250,21.177
4,BY,20,0.984,160.200,35.361
5,BZ,14,0.991,132.000,43.101
6,CX,8,0.926,984.375,16.891
7,CY,15,0.983,255.133,34.542
8,CZ,16,0.995,45.562,57.919


**A-Class (AX, AY, AZ)**
- Lower fill rate (~95–98%), highest lost sales  
- Moderate inventory (DOI ~14–23 days)  

**Insight:**  
High-value SKUs are under-served → need better forecasting and slight increase in safety stock to reduce lost sales.

**B-Class (BX, BY, BZ)**
- Very high fill rate (~98–99%), low lost sales  
- Increasing DOI (~21 → 43 days)  

**Insight:**  
Stable but signs of overstock → can reduce inventory (especially BY, BZ) to free up capital, while maintaining service.

**C-Class (CX, CY, CZ)**
- Very high DOI (especially CZ ~58 days)  
- High fill rate except CX (~93%), highest lost sales  

**Insight:**  
Imbalanced allocation → overstock in CY, CZ but understock in CX → reallocate inventory to improve efficiency and recover lost sales.

**Key Findings**
- The ABC–XYZ analysis reveals a clear imbalance in inventory allocation across SKU groups. High-value A-class items experience lower fill rates and the highest lost sales despite their strong business impact, indicating understocking and insufficient forecasting accuracy.  

- In contrast, several low-value C-class items maintain excessively high inventory coverage, particularly CZ products, suggesting overstock and inefficient capital utilization. Meanwhile, CX items still suffer from relatively high lost sales, highlighting inconsistent inventory allocation within the C category.  

- B-class products generally maintain strong service levels, but increasing inventory days for BY and BZ items may indicate opportunities to reduce excess stock without significantly affecting availability.  

Overall, the results suggest that the current inventory strategy is reactive and not fully aligned with SKU importance and demand variability. A differentiated ABC–XYZ driven inventory policy would improve service levels, reduce unnecessary inventory holding, and support more efficient capital allocation.

## Optimization scope

Instead of optimizing all SKU segments, this project focuses on **high-impact classes** identified through ABC-XYZ segmentation and key metrics (fill rate, lost sales, DOI).

**Selection Criteria**
- **High lost sales** → service gap  
- **High DOI** → overstock  
- **High business impact** → prioritize A-class  

**1. A-Class (AX, AY) – High Impact, Under-served**
- Lower fill rate (~95–98%), highest lost sales (~750–780)  
- Moderate DOI (~14–23 days)  

**Rationale:**  
Critical SKUs but under-served → improving service level yields **high business impact**.

**2. CX – Understocked Segment**
- Lowest fill rate (~93%), highest lost sales (~984)  
- Low DOI (~17 days)  

**Rationale:**  
Clear understock → increasing inventory can **recover lost sales efficiently**.

**3. BZ, CZ – Overstocked Segments**
- Very high DOI (~40–60 days)  
- Near-perfect fill rate (~99%), minimal lost sales  

**Rationale:**  
Overstocked with diminishing returns → opportunity to **reduce inventory and cost**.

**Out-of-Scope (BX, BY, CY)**
- Balanced service and inventory  

**Rationale:**  
Limited optimization potential.

**Optimization Strategy**

| Segment | Key Issue   | Focus |
|--------|------------|------|
| AX, AY | Under-served | Increase safety stock, improve forecast |
| CX     | Understock   | Adjust ROP & safety stock |
| BZ, CZ | Overstock    | Reduce safety stock / order quantity |

**Expected Impact**
- ↑ Service level for **A-class**  
- ↓ Lost sales in **CX**  
- ↓ Holding cost in **BZ, CZ**

## Save class sku

In [80]:
sku_class[['sku_id', 'class']].to_csv(os.path.join(DATA_DIR, "sku_class.csv"), index = False)